## [1] Load & Inspect

In [154]:
import pandas as pd,numpy as np
df = pd.read_csv("../data/raw/food-delivery.csv")
print(df.shape)
print(df.info())
df.head()

(45593, 20)
<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_conditi

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## [2] Clean Text & Empty Strings

In [155]:
# remove spaces from strings
df.columns = df.columns.str.strip()

# remove from text columns 
text_cols = df.select_dtypes(include=['object','str']).columns

for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

# clean weather value (from conditions sunny to sunny)
df['Weatherconditions'] = df['Weatherconditions'].str.replace(r'^conditions\s*','',regex=True)

# clean delivery time Time_taken(min) 
df['Time_taken(min)'] = (
    df['Time_taken(min)']
    .astype('string')
    .str.replace(r'\(min\)', '', regex=True)
    .str.strip()
)

df['Time_taken(min)'] = pd.to_numeric(
    df['Time_taken(min)'],
    errors='coerce'
)

# convert text missing values into nan value
df = df.replace(
    r"^\s*(NaN|nan|None|null)\s*$",np.nan,regex=True
)

# print(df.head())
print(df.isna().sum())

print(df['Time_taken(min)'].head(40))

ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions               616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken(min)                   0
dtype: int64
0     24
1     33
2     26
3     21
4     30
5     26
6     40
7     32
8     34
9     46
10    23
11    21
12    20
13    41
14    20
15    33
16    40
17    41
18    15
19    36
20    26
21    20
22    39
23    34
24    15
25    18
26    38
27    47
28    12
29    

## [3] Convert Columns to the correct types

In [156]:
# rename target name to Time_taken_min
df = df.rename(columns= {'Time_taken(min)':'Delivery_Time_min'})

# numeric Columns 
numeric_cols = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'Restaurant_latitude',
    'Restaurant_longitude',
    'Delivery_location_latitude',
    'Delivery_location_longitude',
    'Vehicle_condition',
    'multiple_deliveries',
    'Delivery_Time_min'
    ]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')

# convert date
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y"
)

# convert time_order , time_order_picked


df["Time_Orderd"] = pd.to_datetime(
    df["Time_Orderd"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

df["Time_Order_picked"] = pd.to_datetime(
    df["Time_Order_picked"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time




print(df.dtypes)

# print("\n\nMessing Values:\n",df.isna().sum())

ID                                     string
Delivery_person_ID                     string
Delivery_person_Age                     Int64
Delivery_person_Ratings               Float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[us]
Time_Orderd                            object
Time_Order_picked                      object
Weatherconditions                      string
Road_traffic_density                   string
Vehicle_condition                       int64
Type_of_order                          string
Type_of_vehicle                        string
multiple_deliveries                     Int64
Festival                               string
City                                   string
Delivery_Time_min                       Int64
dtype: object


## [4] Handle missing values
- Delivery_person_Age have 1854 missing value so the action is to **impute with the median value** because age has outliers & Median is robust and not affected by extreme values.
- Delivery_person_Ratings have 1908 missing value so the action is to **impute with the median value** because Ratings are skewed & Median is safer than mean
- multiple_deliveries have 993 missing value so the action is  to **impute with median value** because It's a count (0, 1, 2…) & Median keeps the distribution realistic.

- **Time_Orderd** has 1731 missing values, so the action is to **impute with the mode value** because time of order is categorical in nature, and the most frequent time is a reasonable replacement for missing values.

- **Weatherconditions** has 616 missing values, so the action is to **impute with the mode value** because weather is a categorical variable, and the most common weather condition is a reasonable replacement.

- **Road_traffic_density** has 601 missing values, so the action is to **impute with the mode value** because traffic density is categorical, and using the most frequent category preserves the existing distribution.

- **Festival** has 228 missing values, so the action is to **impute with the mode value** because Festival is categorical, and the most frequent category is a simple and appropriate replacement.

- **City** has 1200 missing values, so the action is to **impute with the mode value** because City is categorical, and using the most common city is appropriate for missing categorical values.



In [157]:
print("[[[ Missing Value before Handle Missing Value ]]]")
print(df.isna().sum())

# numeric columns -> Median
numeric_cols = ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')
    df[col] = df[col].fillna(df[col].median())

cat_cols = cat_cols = ['Time_Orderd', 'Weatherconditions', 'Road_traffic_density', 'Festival', 'City']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


print("[[[ Missing Value after Handle Missing Value ]]]")
print(df.isna().sum())



# df['Delivery_person_Age'].describe()


[[[ Missing Value before Handle Missing Value ]]]
ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions               616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Delivery_Time_min                 0
dtype: int64
[[[ Missing Value after Handle Missing Value ]]]
ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Res

## [5] Detect and remove invalid coordinates | drop unnecessary columns
Coordinates equal to 0, negative coordinates, or coordinates outside India are invalid for this dataset. Keeping them would create incorrect distances.
dataset contains Indian cities, so we use broad coordinates for India:
Latitude: 6–38
Longitude: 68–98

In [158]:
print('shape before cleaning: ',df.shape)


coords = ['Restaurant_latitude', 'Restaurant_longitude', 
          'Delivery_location_latitude', 'Delivery_location_longitude']





# convert negative coors to positive

print("Negative coordinate values before correction:")
print((df[coords] < 0).sum())
df[coords] = df[coords].abs()

# remove invalid coods with 0
df[coords] = df[coords].replace(0, np.nan)
df = df.dropna(subset=coords)

valid_coordinates = (
    df["Restaurant_latitude"].between(6, 38) &
    df["Restaurant_longitude"].between(68, 98) &
    df["Delivery_location_latitude"].between(6, 38) &
    df["Delivery_location_longitude"].between(68, 98)
)
print("Invalid coordinate rows:", (~valid_coordinates).sum())



df = df.loc[valid_coordinates].copy()








# drop unnecessary columns
df = df.drop(columns=['ID', 'Delivery_person_ID'], errors='ignore')

# drop duplicate 
df = df.drop_duplicates().reset_index(drop=True)

print('shape after cleaning: ',df.shape)





shape before cleaning:  (45593, 20)
Negative coordinate values before correction:
Restaurant_latitude            431
Restaurant_longitude           162
Delivery_location_latitude       0
Delivery_location_longitude      0
dtype: int64
Invalid coordinate rows: 0
shape after cleaning:  (41953, 18)


## Remove logically invalid numerical values

In [159]:
df = df[df['Delivery_person_Age'].between(18, 65)]
df = df[df['Delivery_person_Ratings'].between(1.0, 5.0)]
df = df[df['Delivery_Time_min'].between(5, 120)]
df = df[df['multiple_deliveries'].between(0, 3)]

print('shape after cleaning: ',df.shape)


shape after cleaning:  (41872, 18)


## Save clean data

In [160]:
print("Final Clean Dataset Shape:", df.shape)
df.to_csv("../data/clean/clean_data.csv", index=False)
print("Saved clean data to data/clean/clean_data.csv")

Final Clean Dataset Shape: (41872, 18)
Saved clean data to data/clean/clean_data.csv
